# 02 - 数据聚合

将个体级别的科学家数据，聚合为可视化需要的国家级/学科级 JSON 文件。

## 加载数据
读取 `scientists_raw.csv`，添加派生字段。

In [ ]:
import pandas as pd
import json
import os

# 切换到项目根目录
os.chdir(os.path.dirname(os.getcwd()))

df = pd.read_csv('data/scientists_raw.csv', low_memory=False)
print(f'行数: {len(df)}, 列数: {len(df.columns)}')
df.head(2)

## 派生字段

In [ ]:
# 本土/海外标记
df['is_diaspora'] = (df['cntry'] != 'Greece').astype(int)

# 学术年龄: 2020 - firstyr
df['academic_age'] = 2020 - df['firstyr']

# 百分位分组
def percentile_group(p):
    if pd.isna(p):
        return 'unknown'
    if p <= 1:
        return 'top_1'
    elif p <= 5:
        return 'top_5'
    elif p <= 10:
        return 'top_10'
    else:
        return 'other'

df['percentile_group'] = df['top percentile'].apply(percentile_group)

# 输出目录（chdir 后已在项目根目录）
OUT_DIR = 'data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

print('字段已添加: is_diaspora, academic_age, percentile_group')
print(f'本土: {(df["is_diaspora"]==0).sum()}, 海外: {(df["is_diaspora"]==1).sum()}')

## 1. 按国家聚合 → 世界地图 + Top 10 柱状图

统计每个国家的：科学家总数、海外/本土人数、中位引用、中位 h-index、Top 1% 人数、最主要学科。

In [ ]:
def agg_country(g):
    top1 = g[g['percentile_group'] == 'top_1']
    # 找最主要学科（人数最多的 sm-field）
    top_field = g['sm-field'].value_counts().index[0] if not g['sm-field'].isna().all() else ''
    return pd.Series({
        'total': len(g),
        'overseas': g['is_diaspora'].sum(),
        'domestic': (1 - g['is_diaspora']).sum(),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median(),
        'top_1_count': len(top1),
        'top_field': top_field,
        'scientist_count': len(g)
    })

by_country = df.groupby('cntry').apply(agg_country).reset_index()
by_country = by_country.rename(columns={'cntry': 'country'})
by_country = by_country.sort_values('total', ascending=False)

# 前 10 单独存一份
top10 = by_country.head(10)

by_country.to_json(f'{OUT_DIR}/by_country.json', orient='records', force_ascii=False)
top10.to_json(f'{OUT_DIR}/top10_countries.json', orient='records', force_ascii=False)

print(f'国家总数: {len(by_country)}')
print('Top 10:')
for _, r in top10[['country', 'total', 'overseas_pct']].iterrows():
    print(f'  {r["country"]}: {int(r["total"])} 人 (海外 {r["overseas_pct"]}%)')

## 2. 按学科聚合 → 树图

统计每个大类和子学科的人数、海外占比、Top 1% 人数、中位引用。

In [ ]:
# 大类学科
by_field = df.groupby('sm-field').apply(
    lambda g: pd.Series({
        'scientist_count': len(g),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'top_1_count': (g['percentile_group'] == 'top_1').sum(),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median()
    })
).reset_index()
by_field = by_field.rename(columns={'sm-field': 'field'})
by_field = by_field.sort_values('scientist_count', ascending=False)

by_field.to_json(f'{OUT_DIR}/by_field.json', orient='records', force_ascii=False)
print(f'学科大类数: {len(by_field)}')
by_field.head(10)

In [ ]:
# 子学科（树图用）
by_subfield = df.groupby(['sm-field', 'sm-subfield-1']).apply(
    lambda g: pd.Series({
        'scientist_count': len(g),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'top_1_count': (g['percentile_group'] == 'top_1').sum(),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median(),
        'domestic_count': (1 - g['is_diaspora']).sum(),
        'overseas_count': g['is_diaspora'].sum()
    })
).reset_index()
by_subfield = by_subfield.rename(columns={'sm-field': 'field', 'sm-subfield-1': 'subfield'})
by_subfield = by_subfield.sort_values('scientist_count', ascending=False)

by_subfield.to_json(f'{OUT_DIR}/by_subfield.json', orient='records', force_ascii=False)
print(f'子学科数: {len(by_subfield)}')

## 3. 国家×学科矩阵 → 热力矩阵

行是国家，列是学科大类，值为科学家数量或 Top 1% 人数。

In [ ]:
matrix = df.pivot_table(
    index='cntry',
    columns='sm-field',
    values='authfull',
    aggfunc='count'
).fillna(0).astype(int)

# 只保留科学家总数 >= 50 的国家
country_totals = df['cntry'].value_counts()
major_countries = country_totals[country_totals >= 50].index
matrix = matrix[matrix.index.isin(major_countries)]

result = {
    'countries': list(matrix.index),
    'fields': list(matrix.columns),
    'values': matrix.values.tolist()
}

with open(f'{OUT_DIR}/country_field_matrix.json', 'w') as f:
    json.dump(result, f, ensure_ascii=False)

print(f'矩阵维度: {len(result["countries"])} 国 × {len(result["fields"])} 学科')
print(f'国家: {result["countries"][:5]}...')

## 4. 本土/海外对比 → 散点图 + 盒须图

对比本土和海外科学家的引用、h-index、发文量分布。

In [ ]:
def percentile_buckets(series, n=20):
    """计算分布的分位点，用于箱线图/山脊线图"""
    buckets = []
    for i in range(n + 1):
        buckets.append(round(series.quantile(i / n), 1))
    return buckets

def agg_diaspora(g):
    return pd.Series({
        'count': len(g),
        'median_np': g['np'].median(),
        'median_nc': g['nc9619'].median(),
        'median_h': g['h19'].median(),
        'mean_np': round(g['np'].mean(), 1),
        'mean_nc': round(g['nc9619'].mean(), 1),
        'mean_h': round(g['h19'].mean(), 1),
        'np_percentiles': percentile_buckets(g['np']),
        'nc_percentiles': percentile_buckets(g['nc9619']),
        'h_percentiles': percentile_buckets(g['h19'])
    })

diaspora_grp = df.groupby('is_diaspora').apply(agg_diaspora).reset_index()
diaspora_grp['group'] = diaspora_grp['is_diaspora'].map({0: 'domestic', 1: 'overseas'})

# 也按学科细分
diaspora_field = df.groupby(['is_diaspora', 'sm-field']).apply(
    lambda g: pd.Series({
        'count': len(g),
        'median_nc': g['nc9619'].median(),
        'median_h': g['h19'].median()
    })
).reset_index()

diaspora_grp.to_json(f'{OUT_DIR}/diaspora_comparison.json', orient='records', force_ascii=False)
diaspora_field.to_json(f'{OUT_DIR}/diaspora_by_field.json', orient='records', force_ascii=False)

print('本土 vs 海外 整体对比:')
for _, r in diaspora_grp.iterrows():
    print(f'  {r["group"]}: {int(r["count"])} 人 | 中位发文 {r["median_np"]} | 中位引用 {r["median_nc"]} | 中位h {r["median_h"]}')

## 5. 顶尖人才迁移

Top 1% 和 Top 5% 科学家的国家分布。

In [ ]:
def agg_talent(g):
    return pd.Series({
        'total': len(g),
        'overseas': g['is_diaspora'].sum(),
        'domestic': (1 - g['is_diaspora']).sum(),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2)
    })

top_talent = df[df['percentile_group'].isin(['top_1', 'top_5'])].copy()
top_talent_by_country = top_talent.groupby('cntry').apply(agg_talent).reset_index()
top_talent_by_country = top_talent_by_country.rename(columns={'cntry': 'country'})
top_talent_by_country = top_talent_by_country.sort_values('total', ascending=False)

top_talent_by_country.to_json(f'{OUT_DIR}/top_talent_by_country.json', orient='records', force_ascii=False)
print(f'Top 1%/5% 科学家总数: {len(top_talent)}')
print(f'海外占比: {round(top_talent["is_diaspora"].mean() * 100, 1)}%')
top_talent_by_country.head(10)

## 6. 学术年龄分析 → 下一代科学家

按学术年龄分组，看不同年龄段的本土/海外分布趋势。

In [ ]:
# 学术年龄分桶
df['age_group'] = pd.cut(
    df['academic_age'],
    bins=[0, 5, 10, 15, 20, 25, 30, 35, 100],
    labels=['0-5', '6-10', '11-15', '16-20', '21-25', '26-30', '31-35', '36+']
)

age_analysis = df.groupby('age_group').apply(
    lambda g: pd.Series({
        'total': len(g),
        'overseas': g['is_diaspora'].sum(),
        'domestic': (1 - g['is_diaspora']).sum(),
        'overseas_pct': round(g['is_diaspora'].mean() * 100, 2),
        'median_citation': g['nc9619'].median(),
        'median_hindex': g['h19'].median()
    })
).reset_index()

# 也按学术年龄+海外拆分的科学家数量
age_diaspora = df.groupby(['age_group', 'is_diaspora']).size().reset_index(name='count')
age_diaspora['group'] = age_diaspora['is_diaspora'].map({0: 'domestic', 1: 'overseas'})

age_analysis.to_json(f'{OUT_DIR}/academic_age.json', orient='records', force_ascii=False)
age_diaspora.to_json(f'{OUT_DIR}/age_diaspora.json', orient='records', force_ascii=False)

print('各年龄段海外占比:')
for _, r in age_analysis.iterrows():
    print(f'  {r["age_group"]}: {int(r["total"])} 人, 海外 {r["overseas_pct"]}%')

## 7. 全局概览统计

给 Overview Cards 用的全局指标。

In [ ]:
overview = {
    'total_scientists': len(df),
    'total_countries': df['cntry'].nunique(),
    'overseas_pct': round(df['is_diaspora'].mean() * 100, 2),
    'domestic_count': (df['is_diaspora'] == 0).sum(),
    'overseas_count': df['is_diaspora'].sum(),
    'top_1_count': (df['percentile_group'] == 'top_1').sum(),
    'median_citation': round(df['nc9619'].median(), 1),
    'median_hindex': round(df['h19'].median(), 1),
    'total_fields': df['sm-field'].nunique(),
    'total_subfields': df['sm-subfield-1'].nunique()
}

with open(f'{OUT_DIR}/overview_stats.json', 'w') as f:
    json.dump(overview, f, ensure_ascii=False)

for k, v in overview.items():
    print(f'{k}: {v}')

## 8. 导出散点图采样 & 复制到 web/data/

为 D3 前端生成采样数据（全量 6 万点浏览器扛不住），并将所有 JSON 复制到 web/data/。

In [ ]:
import random
import shutil

# 采样 3000 个点用于散点图
sample_df = df[(df['np'] > 0) & (df['nc9619'] > 0)].sample(n=min(3000, len(df)), random_state=42)
scatter_sample = sample_df[['np', 'nc9619', 'is_diaspora']].to_dict(orient='records')
with open(f'{OUT_DIR}/scatter_sample.json', 'w') as f:
    json.dump(scatter_sample, f, ensure_ascii=False)

# 复制到 web/data/
WEB_DATA_DIR = 'web/data'
os.makedirs(WEB_DATA_DIR, exist_ok=True)
for fname in os.listdir(OUT_DIR):
    if fname.endswith('.json'):
        shutil.copy2(os.path.join(OUT_DIR, fname), os.path.join(WEB_DATA_DIR, fname))

print('已复制到 web/data/:')
for f in sorted(os.listdir(WEB_DATA_DIR)):
    size = os.path.getsize(os.path.join(WEB_DATA_DIR, f))
    print(f'  {f:35s} {size/1024:>8.1f} KB')